# Submit cluster-side QC analysis (SLURM)

Run **on a cluster node** with `sbatch`/`sacct`, with `SAMPLE_DIR` where the
transferred data is. Data still arriving is fine: it acts only on files that
exist.

For each pending round it:
1. Lists the round's not-yet-analysed FOV files.
2. Submits a SLURM array running `analysis/cli_analyze_fov.py` per FOV (as
   `FOVScheduler` does locally).
3. When all FOVs are done, submits `analysis/cli_build_round_mosaic.py` (as
   `RoundScheduler` does).

`.fov_submitted`/`.round_mosaic_submitted` sentinels hold the job id, so work
still `PENDING`/`RUNNING` is not resubmitted.

## 1 — Setup

In [ ]:
import os
import sys
import time
import logging
from pathlib import Path

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/after_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root on the cluster
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.progress        import ProgressTracker
from MERci.acquisition.cluster_submit import run_submission_pass

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Parameters

Set `DRY_RUN = True` the first time, to see what would be submitted.

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)

# ── Image file format (must match what HAL wrote) ────────────────────────────
IMAGE_SUFFIX = ".zarr"   # options: ".zarr", ".dax", ".tiff"

# ── SLURM job parameters ──────────────────────────────────────────────────
PARTITION         = "zhuang,sapphire,shared"
CONDA_ENV         = "merci_env"   # cluster env with environment.yml's dependencies
ARRAY_CONCURRENCY = 50            # max FOV-analysis array tasks running at once
FOV_MEM,    FOV_TIME    = "8gb",  "02:00:00"
MOSAIC_MEM, MOSAIC_TIME = "8gb",  "00:30:00"

# ── Behaviour ──────────────────────────────────────────────────────────────
DRY_RUN       = True    # True: build scripts + print what would be submitted, but don't call sbatch
POLL_INTERVAL = 1800    # seconds between passes if you run the continuous loop in section 5

print(f"Sample name  : {SAMPLE_NAME}")
print(f"Positions tag: {POSITIONS_TAG}")
print(f"Image suffix : {IMAGE_SUFFIX}")
print(f"Partition    : {PARTITION}")
print(f"Conda env    : {CONDA_ENV}")
print(f"Dry run      : {DRY_RUN}")

In [ ]:
config = ExperimentConfig.from_sample_dir(
    SAMPLE_DIR,
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
)

meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)
tracker = ProgressTracker(config.analysis_dir)

MANIFESTS_DIR = config.analysis_dir / "logs"
MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Rounds     : {meta.n_rounds}")
print(f"FOVs       : {meta.n_fovs}")
print(f"Manifests  : {MANIFESTS_DIR}")

## 3 — Check current progress

In [ ]:
summary = tracker.summary(meta)
print(f"FOVs  done : {summary['files_fov_done']} / {summary['files_total']}")
print(f"Rounds done: {summary['rounds_done']} / {summary['rounds_total']}")

## 4 — Submission functions

In `MERci.acquisition.cluster_submit` (`run_submission_pass`).

## 5 — Run one pass

Re-run whenever more data has arrived. Safe to run as often as you like.

In [ ]:
result = run_submission_pass(
    config, meta, tracker, SAMPLE_DIR, MANIFESTS_DIR,
    ARRAY_CONCURRENCY, FOV_MEM, FOV_TIME, MOSAIC_MEM, MOSAIC_TIME,
    PARTITION, CONDA_ENV, dry_run=DRY_RUN,
)
summary = result["summary"]
print(f"FOV job(s) submitted this pass   : {result['fov_submitted']}")
print(f"Mosaic job(s) submitted this pass: {result['mosaic_submitted']}")
print(f"FOVs  done : {summary['files_fov_done']} / {summary['files_total']}")
print(f"Rounds done: {summary['rounds_done']} / {summary['rounds_total']}")

## 6 — Optional: run continuously

Polls instead of re-running section 5 by hand. Use `tmux`/`nohup` to keep it
running after closing the notebook. Interrupt the kernel (`■`) to stop.

In [ ]:
iteration = 0
while True:
    print(f"=== tick {iteration} ===")
    result = run_submission_pass(
        config, meta, tracker, SAMPLE_DIR, MANIFESTS_DIR,
        ARRAY_CONCURRENCY, FOV_MEM, FOV_TIME, MOSAIC_MEM, MOSAIC_TIME,
        PARTITION, CONDA_ENV, dry_run=DRY_RUN,
    )
    summary = result["summary"]
    print(f"FOV job(s) submitted   : {result['fov_submitted']}")
    print(f"Mosaic job(s) submitted: {result['mosaic_submitted']}")
    print(f"FOVs  done : {summary['files_fov_done']} / {summary['files_total']}")
    print(f"Rounds done: {summary['rounds_done']} / {summary['rounds_total']}")
    iteration += 1
    time.sleep(POLL_INTERVAL)